# Plan C — Direct VLM on Page Images (No OCR Text)

Skips Tesseract text entirely. Feeds the page PNGs directly to a vision-language model via Ollama — one page at a time. The VLM reads layout, typography, and content simultaneously, bypassing OCR error propagation.

**Best for:** poor OCR quality, complex multi-column layouts, scanned documents.

**Full pipeline:**
1. Install system dependencies (`poppler-utils` for `pdftoppm` — no Tesseract needed)
2. Convert PDF to page PNGs via `pdftoppm`
3. Page triage — classify each page as catalog vs. non-catalog with a fast VLM
4. Per-page structured extraction on flagged pages
5. Cross-page merge (entries spanning two pages)
6. Local entity resolution
7. Write QuickStatements-compatible CSV + human-readable CSV

**Triage and extraction results are cached incrementally** — safe to interrupt and re-run.

In [ ]:
# ── Parameters ──────────────────────────────────────────────────────────────
# Set these before running. All other paths are derived from pdf_file + output_dir.

pdf_file          = "baldessari-sprengel.pdf"   # input PDF
pdf_dpi           = 300                         # resolution for PDF → PNG conversion
output_dir        = "."                         # directory for all outputs
triage_model      = "qwen2.5vl:7b"             # fast model for YES/NO page triage
extraction_model  = "qwen3-vl:8b"              # stronger model for structured extraction

## Install Dependencies

Plan C only needs `poppler-utils` (for `pdftoppm`) — no Tesseract required. The `ollama` Python package is also installed if missing.

In [ ]:
import shutil
import subprocess
import sys

def apt_install(*packages):
    subprocess.run(
        ["apt-get", "install", "-y", "--no-install-recommends"] + list(packages),
        check=True,
    )

if shutil.which("pdftoppm"):
    print("  pdftoppm: OK")
else:
    print("  pdftoppm: missing — installing poppler-utils")
    apt_install("poppler-utils")

try:
    import ollama
    print("  ollama (Python package): OK")
except ImportError:
    print("  ollama (Python package): installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "ollama"], check=True)
    import ollama

r = subprocess.run(["pdftoppm", "-v"], capture_output=True, text=True)
print((r.stdout or r.stderr).splitlines()[0])

In [ ]:
import re
import json
import csv
import difflib
import time
from pathlib import Path
import ollama

# Derive all paths from parameters
pdf_path  = Path(pdf_file)
stem      = pdf_path.stem
_out      = Path(output_dir)
_out.mkdir(parents=True, exist_ok=True)

PAGES_DIR       = _out / f"{stem}_pages"
OUTPUT_CSV      = _out / "output-plan-c.csv"
OUTPUT_READABLE = _out / "output-plan-c-readable.csv"
OUTPUT_FAILURES = _out / "output-plan-c-failures.txt"
TRIAGE_CACHE    = _out / "output-plan-c-triage.json"
EXTRACT_CACHE   = _out / "output-plan-c-raw.json"

print(f"PDF        : {pdf_path}  (exists: {pdf_path.exists()})")
print(f"Pages dir  : {PAGES_DIR}")
print(f"Output dir : {_out.resolve()}")

# ── Entity resolution lookup tables ─────────────────────────────────────────
CREATOR_QIDS = {
    "John Baldessari": "Q312793",
    "Baldessari": "Q312793",
}
OBJECT_TYPE_QIDS = {
    "painting": "Q3305213",
    "photograph": "Q125191",
    "gelatin silver print": "Q3286805",
    "video": "Q34508",
    "drawing": "Q93184",
    "print": "Q11060274",
    "collage": "Q170571",
    "mixed media": "Q1902763",
}
MATERIAL_QIDS = {
    "acrylic": "Q207849",
    "oil": "Q296955",
    "canvas": "Q4259259",
    "gelatin silver": "Q3286805",
    "photograph": "Q125191",
    "ink": "Q127418",
    "pencil": "Q14674",
    "paper": "Q11472",
}

## PDF → Page Images

Converts each PDF page to a PNG at the specified DPI using `pdftoppm`. Skip this cell if the images already exist.

In [ ]:
PAGES_DIR.mkdir(exist_ok=True)

existing = sorted(PAGES_DIR.glob("page-*.png"))
if existing:
    print(f"Found {len(existing)} existing page images — skipping conversion.")
    print(f"  Delete {PAGES_DIR} to re-run from scratch.")
else:
    print(f"Converting {pdf_path.name} to PNG at {pdf_dpi} DPI...")
    result = subprocess.run(
        ["pdftoppm", "-r", str(pdf_dpi), "-png", str(pdf_path), str(PAGES_DIR / "page")],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"pdftoppm stderr: {result.stderr}")
        raise RuntimeError("pdftoppm failed")
    existing = sorted(PAGES_DIR.glob("page-*.png"))
    print(f"Created {len(existing)} page images.")

pages = sorted(
    int(re.search(r"page-(\d+)\.png", p.name).group(1))
    for p in existing
)
print(f"Pages: {pages[0]:02d} to {pages[-1]:02d}  ({len(pages)} total)")

## Step 0 — OCR Quality Note

Plan C bypasses OCR entirely — no Tesseract, no aspell quality gate needed. This step just confirms we're working from images and reports the page count.

In [ ]:
print("=== Step 0: OCR Quality Note ===")
print(f"Plan C reads {len(pages)} page images directly via VLM.")
print("No OCR text is used — Tesseract output is not required.")
print(f"Triage model:    {triage_model}")
print(f"Extraction model: {extraction_model}")

## Step 1 — Page Triage

A fast VLM (`qwen2.5vl:7b`) classifies each page as YES (contains catalog entries) or NO. Results are cached incrementally — safe to interrupt and re-run.

In [ ]:
ALLOWED_VISION_MODELS = ("qwen3-vl", "qwen2.5vl", "llava", "llama3.2-vision")


def wait_for_ollama_idle():
    while True:
        result = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
        lines  = result.stdout.strip().split("\n")[1:]
        active = [l for l in lines if l.strip() and "Stopping" not in l]
        if not active or all(any(m in l for m in ALLOWED_VISION_MODELS) for l in active):
            break
        print("Waiting for text LLM to finish...", flush=True)
        time.sleep(20)


def step1_triage(pages):
    print("=== Step 1: Page Triage ===")
    triage = {}
    if TRIAGE_CACHE.exists():
        try:
            triage = {int(k): v for k, v in json.loads(TRIAGE_CACHE.read_text()).items()}
            print(f"  Loaded triage cache: {len(triage)} pages already classified.")
        except Exception:
            triage = {}

    if all(n in triage for n in pages):
        flagged = [n for n, is_cat in sorted(triage.items()) if is_cat]
        print(f"Cache complete: {len(flagged)} of {len(pages)} pages flagged.")
        return triage

    prompt = (
        "Does this page contain structured artwork catalog entries listing specific artworks "
        "with title, date, materials, or dimensions? Answer with exactly one word: YES or NO."
    )

    for n in pages:
        if n in triage:
            print(f"  Page {n:02d}: cached -> {'YES' if triage[n] else 'NO'}")
            continue
        img_path = PAGES_DIR / f"page-{n:02d}.png"
        print(f"  Triaging page {n:02d}...", end=" ", flush=True)
        wait_for_ollama_idle()
        try:
            resp   = ollama.chat(
                model=triage_model,
                messages=[{"role": "user", "content": prompt, "images": [str(img_path)]}],
            )
            out    = resp.message.content.strip()
            triage[n] = "yes" in out.lower()
            print(f"-> {'YES' if triage[n] else 'NO'}  ({out[:60]!r})")
        except Exception as e:
            print(f"-> ERROR: {e}, marking NO")
            triage[n] = False
        TRIAGE_CACHE.write_text(json.dumps({str(k): v for k, v in triage.items()}, indent=2))

    TRIAGE_CACHE.write_text(json.dumps({str(k): v for k, v in triage.items()}, indent=2))
    flagged = [n for n, is_cat in sorted(triage.items()) if is_cat]
    print(f"\nTriage complete: {len(flagged)} of {len(pages)} pages flagged.")
    print(f"Flagged pages: {flagged}")
    return triage

In [ ]:
triage = step1_triage(pages)

## Step 2 — Per-Page Structured Extraction

Each page flagged YES is sent to the extraction VLM (`qwen3-vl:8b`). Results are cached after each page — safe to interrupt and re-run.

In [ ]:
def parse_json_array(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"\s*```\s*$",       "", text, flags=re.MULTILINE)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    try:
        data = json.loads(text)
        if isinstance(data, list): return data
    except json.JSONDecodeError:
        pass
    m = re.search(r"\[.*\]", text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group(0))
            if isinstance(data, list): return data
        except json.JSONDecodeError:
            pass
    objects = re.findall(r"\{[^{}]*\}", text, re.DOTALL)
    if objects:
        try: return [json.loads(o) for o in objects]
        except json.JSONDecodeError: pass
    return None


def step2_extract(triage, pages):
    print("=== Step 2: Per-Page Structured Extraction ===")
    flagged = [n for n in pages if triage.get(n, False)]
    print(f"Extracting from {len(flagged)} flagged pages...")

    failures    = []
    all_records = []
    completed_pages = set()

    if EXTRACT_CACHE.exists():
        try:
            cached = json.loads(EXTRACT_CACHE.read_text())
            if isinstance(cached, list):
                all_records     = cached
                completed_pages = {r.get("source_page") for r in all_records if r.get("source_page")}
                print(f"  Loaded extraction cache: {len(completed_pages)} pages done, {len(all_records)} records.")
        except Exception:
            pass

    if all(n in completed_pages for n in flagged):
        print(f"Cache complete: {len(all_records)} raw records.")
        return all_records

    prompt = (
        "Extract all artwork catalog entries visible on this page. "
        "Return ONLY a valid JSON array. Each element is one artwork entry with these fields: "
        "title (string), creator (string), date (string or year), object_type (string), "
        "materials (array of strings), height_cm (number or null), width_cm (number or null), "
        "depth_cm (number or null), inventory_number (string or null), catalog_number (string or null). "
        "If you see '120 x 95 cm' parse height as 120, width as 95. "
        "Use null for any field not clearly visible. Do NOT invent values. "
        "Return [] if no catalog entries are found."
    )

    for n in flagged:
        if n in completed_pages:
            count = sum(1 for r in all_records if r.get("source_page") == n)
            print(f"  Page {n:02d}: cached -> {count} records")
            continue
        img_path = PAGES_DIR / f"page-{n:02d}.png"
        print(f"  Extracting page {n:02d}...", end=" ", flush=True)
        wait_for_ollama_idle()
        try:
            resp    = ollama.chat(
                model=extraction_model,
                messages=[{"role": "user", "content": prompt, "images": [str(img_path)]}],
            )
            out     = resp.message.content.strip()
            records = parse_json_array(out)
            if records is None:
                print("-> PARSE FAILURE")
                failures.append(f"Page {n:02d}: could not parse JSON\nRaw output:\n{out[:500]}\n---")
                records = []
            else:
                for rec in records:
                    rec["source_page"] = n
                all_records.extend(records)
                print(f"-> {len(records)} records")
        except Exception as e:
            print(f"-> ERROR: {e}")
            failures.append(f"Page {n:02d}: {e}\n---")
        EXTRACT_CACHE.write_text(json.dumps(all_records, indent=2, ensure_ascii=False))

    if failures:
        OUTPUT_FAILURES.write_text("\n".join(failures))
        print(f"\n{len(failures)} failures logged to {OUTPUT_FAILURES}")
    else:
        OUTPUT_FAILURES.write_text("No extraction failures.\n")

    EXTRACT_CACHE.write_text(json.dumps(all_records, indent=2, ensure_ascii=False))
    print(f"\nExtraction complete: {len(all_records)} raw records from {len(flagged)} pages.")
    return all_records

In [ ]:
raw_records = step2_extract(triage, pages)

## Step 3 — Cross-Page Merge

Entries that span two pages (e.g. title on page N, dimensions on page N+1) are detected by matching `catalog_number` or high title similarity and merged.

In [ ]:
def title_similarity(a, b):
    if not a or not b: return 0.0
    return difflib.SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio()


def merge_records(a, b):
    merged = {}
    for k in set(a) | set(b):
        va, vb = a.get(k), b.get(k)
        merged[k] = va if (va is not None and va != "" and va != []) else vb
    pages_a = a.get("merged_from_pages", [a.get("source_page")])
    pages_b = b.get("merged_from_pages", [b.get("source_page")])
    merged["merged_from_pages"] = sorted(set((pages_a or []) + (pages_b or [])))
    return merged


def step3_merge(raw_records):
    print("=== Step 3: Cross-Page Merge ===")
    print(f"Starting with {len(raw_records)} raw records.")
    records = sorted(
        raw_records,
        key=lambda r: (r.get("source_page") or 0, str(r.get("catalog_number") or "")),
    )
    merged, used = [], set()
    for i, rec in enumerate(records):
        if i in used: continue
        current = {**rec, "merged_from_pages": [rec.get("source_page")]}
        for j, other in enumerate(records):
            if j <= i or j in used: continue
            cn_i = str(rec.get("catalog_number") or "").strip()
            cn_j = str(other.get("catalog_number") or "").strip()
            is_dup = (cn_i and cn_j and cn_i == cn_j) or \
                     title_similarity(rec.get("title"), other.get("title")) > 0.85
            if is_dup:
                current = merge_records(current, other)
                used.add(j)
        used.add(i)
        merged.append(current)
    print(f"After merge: {len(merged)} records.")
    return merged

In [ ]:
merged_records = step3_merge(raw_records)

## Step 4 — Local Entity Resolution

Fuzzy-match creator names, material terms, and object types to Wikidata Q-IDs using `difflib`. No external APIs.

In [ ]:
def resolve_entity(value, qid_map):
    if not value: return None
    value_lower = value.lower().strip()
    for k, v in qid_map.items():
        if k.lower() == value_lower: return v
    matches = difflib.get_close_matches(value, qid_map.keys(), n=1, cutoff=0.8)
    if matches: return qid_map[matches[0]]
    for k, v in qid_map.items():
        if k.lower() in value_lower or value_lower in k.lower(): return v
    return None


def format_date_qs(date_str):
    if not date_str: return None
    m = re.search(r"\b(1[89]\d\d|20[012]\d)\b", str(date_str))
    return f"+{m.group(1)}-00-00T00:00:00Z/9" if m else None


def step4_entity_resolution(records):
    print("=== Step 4: Entity Resolution ===")
    enriched = []
    for rec in records:
        r = dict(rec)
        r["creator_qid"]     = resolve_entity(r.get("creator"), CREATOR_QIDS) or "Q312793"
        r["object_type_qid"] = resolve_entity(r.get("object_type", ""), OBJECT_TYPE_QIDS)
        mats = r.get("materials") or []
        if isinstance(mats, str): mats = [m.strip() for m in mats.split(",")]
        r["materials"]     = mats
        r["materials_qids"] = [resolve_entity(m, MATERIAL_QIDS) for m in mats]
        r["date_qs"]        = format_date_qs(r.get("date"))
        enriched.append(r)
    print(f"Entity resolution complete for {len(enriched)} records.")
    return enriched

In [ ]:
enriched_records = step4_entity_resolution(merged_records)

## Step 5 — Write Outputs

Writes two CSVs:
- `output-plan-c.csv` — QuickStatements-compatible (Wikidata P-number headers)
- `output-plan-c-readable.csv` — human-readable headers, includes `source_pages`

In [ ]:
def step5_output(records):
    print("=== Step 5: Writing Output CSVs ===")

    qs_fields = ["qid", "P31", "P1476", "P170", "P571", "P276", "P195",
                 "P186", "P2048", "P2049", "P2610", "P217", "P528", "P1431"]

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=qs_fields)
        writer.writeheader()
        for r in records:
            mat_qids     = r.get("materials_qids") or []
            first_mat_qid = next((q for q in mat_qids if q), None)
            title         = r.get("title") or ""
            writer.writerow({
                "qid":   "CREATE",
                "P31":   r.get("object_type_qid"),
                "P1476": f'"{title}"@en' if title else None,
                "P170":  r.get("creator_qid"),
                "P571":  r.get("date_qs"),
                "P276":  "Q673907",
                "P195":  "Q673907",
                "P186":  first_mat_qid,
                "P2048": r.get("height_cm"),
                "P2049": r.get("width_cm"),
                "P2610": r.get("depth_cm"),
                "P217":  r.get("inventory_number"),
                "P528":  r.get("catalog_number"),
                "P1431": "Q138573075",
            })
    print(f"Written: {OUTPUT_CSV}  ({OUTPUT_CSV.stat().st_size:,} bytes)")

    readable_fields = [
        "title", "creator", "creator_qid", "date", "object_type", "object_type_qid",
        "materials", "materials_qids", "height_cm", "width_cm", "depth_cm",
        "inventory_number", "catalog_number", "location", "exhibited_at",
        "source_pages", "notes",
    ]

    with open(OUTPUT_READABLE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=readable_fields)
        writer.writeheader()
        for r in records:
            mats     = r.get("materials") or []
            mat_qids = r.get("materials_qids") or []
            pages_src = r.get("merged_from_pages") or [r.get("source_page")]
            writer.writerow({
                "title":           r.get("title") or "",
                "creator":         r.get("creator") or "John Baldessari",
                "creator_qid":     r.get("creator_qid") or "Q312793",
                "date":            r.get("date") or "",
                "object_type":     r.get("object_type") or "",
                "object_type_qid": r.get("object_type_qid") or "",
                "materials":       "; ".join(str(m) for m in mats) if mats else "",
                "materials_qids":  "; ".join(str(q) for q in mat_qids if q) if mat_qids else "",
                "height_cm":       r.get("height_cm") or "",
                "width_cm":        r.get("width_cm")  or "",
                "depth_cm":        r.get("depth_cm")  or "",
                "inventory_number": r.get("inventory_number") or "",
                "catalog_number":   r.get("catalog_number")   or "",
                "location":        "Sprengel Museum Hannover",
                "exhibited_at":    "Sprengel Museum Hannover",
                "source_pages":    "; ".join(str(p) for p in pages_src if p is not None),
                "notes":           "",
            })
    print(f"Written: {OUTPUT_READABLE}  ({OUTPUT_READABLE.stat().st_size:,} bytes)")


step5_output(enriched_records)

# ── Final report ─────────────────────────────────────────────────────────────
flagged_count = sum(1 for v in triage.values() if v)
print()
print("=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(f"Pages triaged:           {len(pages)} total, {flagged_count} flagged")
print(f"Raw records extracted:   {len(raw_records)}")
print(f"After cross-page merge:  {len(merged_records)}")
for f in [OUTPUT_CSV, OUTPUT_READABLE, OUTPUT_FAILURES]:
    if f.exists():
        print(f"{f.name}: {f.stat().st_size:,} bytes")
print("Done.")
print("=" * 60)